In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("All Libraries imported successfully")

In [ ]:
df = sns.load_dataset('titanic')
print("Dataset loaded successfully")

In [ ]:
df.head()
#df.shape
#df.columns
#df.info()
#df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
plt.Figure(figsize=(10,6))
sns.heatmap(df.isnull(),yticklabels=False,cbar=False,cmap='viridis')
plt.title("Missing values heatmap")
plt.show()

In [ ]:
df['age']= df['age'].fillna(df['age'].median())
df['embarked']=df['embarked'].fillna(df['embarked'].mode()[0])
df=df.drop(columns=['deck','embark_town','alive','who','adult_male', 'alone'])
print(df.isnull().sum())



## Exploratory Data Analysis

##### Chart 1 — How many survived?

In [ ]:
plt.Figure(figsize=(6,4))
sns.countplot(x='survived',data=df)
plt.title("Survival count")
plt.xticks([0,1],['Died','Survived'])
plt.savefig('images/survival_count.png', bbox_inches='tight')
plt.show()
print(df['survived'].value_counts())
print(df['survived'].value_counts(normalize=True)*100)

#### Chart 2 — Survival by Gender

In [ ]:
plt.Figure(figsize=(6,4))
sns.countplot(x='sex',data=df,hue='survived')
plt.title("Survival by Gender")
plt.legend(labels=['Died','Survived'],title='Outcome')
plt.savefig('images/survival_by_gender.png', bbox_inches='tight')
plt.show()

print(df['sex'].value_counts())
print(df['sex'].value_counts(normalize=True)*100)

# percentage breakdown
total=df.groupby('sex')['survived'].agg(['sum','count'])
total['percentage'] = (total['sum']/total['count']*100).round(1)
total.columns=['Survived','Total','Survival Rate %']
print(total)


#### Chart 3 — Survival by Passenger Class

In [ ]:
plt.Figure(figsize=(6,4))
sns.countplot(x='pclass',hue='survived',data=df)
plt.title('Survival by Passenger class')
plt.legend(labels=['Died','Survived'],title='outcome')
plt.savefig('images/survival_by_class.png', bbox_inches='tight')
plt.show


#### Chart 4 — Age Distribution


In [ ]:
plt.Figure(figsize=(8,4))
sns.histplot(df['age'],bins=30,kde=True)
plt.title("Age distribution of passengers")
plt.savefig('images/age_distribution.png', bbox_inches='tight')
plt.show()

#### Chart 5 — Age vs Survival

In [ ]:
plt.Figure(figsize=(8,4))
sns.histplot(data=df,x='age',hue='survived',bins=30,kde=True)
plt.title("Age vs Survival")
plt.legend(labels=['Died','Survived'])
plt.show()

#### Chart 6 — Fare Distribution

In [ ]:
plt.figure(figsize=(8,4))
sns.histplot(data=df['fare'],bins=40,kde=True)
plt.title('Fare Distribution')
plt.savefig('images/fare_distribution.png', bbox_inches='tight')
plt.show()

#### Chart 7 — Correlation Heatmap

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(df.corr(numeric_only=True),annot=True,cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.savefig('images/correlation_heatmap.png', bbox_inches='tight')
plt.show()

#### Chart 8 — Survival by Embarked Port

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='embarked', hue= 'survived',data=df)
plt.title('Survival by Port of Embarkation')
plt.legend(labels=['Died','Survived'])
plt.xticks([0,1,2],['Southampton', 'Cherbourg', 'Queenstown'])
plt.xlabel('Port of Embarkation')
plt.savefig('images/survival_by_port.png',bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='sex', hue='survived', data=df)
plt.title('Survival by Gender')
plt.legend(labels=['Died', 'Survived'])
plt.xlabel('Gender')
plt.ylabel('Count')
plt.show()

### Key Findings

In [ ]:
print('='*40)
print('KEY FINDINGS')
print('='*40)

print(f'\n1.Overall survival rate : {df['survived'].mean()*100:.1f} %')

gender_survival = df.groupby('sex')['survived'].mean()*100
print(f'\n2. Female survival rate : {gender_survival['female']:.1f}')
print(f'   Male survival rate : {gender_survival['male']:.1f}')

class_survival = df.groupby('pclass')['survived'].mean()*100
print('\n')
print(f'3. 1st class survival : {class_survival[1]:.1f} %')
print(f'   2nd class survival : {class_survival[2]:.1f} %')
print(f'   3rd class survival : {class_survival[3]:.1f} %')

children=df[df['age']<10]['survived'].mean() * 100
print(f'\n4. Children (under 10) survival rate : {children:.1f} %')



### Simple ML Model


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

#prepare data
df_ml=df.copy()
df_ml['sex']=df_ml['sex'].map({'male':0,'female': 1})
df_ml['embarked']=df_ml['embarked'].map({'S':0,'C':1,'Q':2})

# Feature and target
x = df_ml[['pclass','sex','age','sibsp','parch','fare']]
y=df_ml['survived']

#split 
x_train, x_test, y_train,y_test = train_test_split(
    x,y,test_size=0.2,random_state=42
)

#train
model = RandomForestClassifier(n_estimators=100,random_state=42)
model.fit(x_train,y_train)

#evaluate
predictions = model.predict(x_test)
accuracy = accuracy_score(y_test,predictions)
print(f"Model Accuracy : {accuracy*100:.2f}%")
print("\nDetailed Report : ")
print(classification_report(y_test,predictions))
